In [ ]:
"""Credit Card Fraud Detection model pipeline.

Requirements addressed:
- Handle class imbalance with SMOTE / undersampling + class weights
- Engineer features from Time and Amount
- Train and compare multiple models
- Evaluate with ROC-AUC, PR-AUC, threshold tuning, and cost-sensitive metrics
- Save the best model to disk

Usage:
    python train_fraud_model.py --data creditcard.csv --outdir artifacts

The public Kaggle dataset used by the original notebook must be downloaded separately.
"""

from __future__ import annotations

import argparse
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple, Any, Optional

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


RANDOM_STATE = 42


class FraudFeatureEngineer(BaseEstimator, TransformerMixin):
    """Add simple high-value features for the fraud dataset.

    The original dataset has anonymized PCA-like features V1..V28 plus
    'Time' and 'Amount'. We only engineer from available semantic columns.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        if not isinstance(df, pd.DataFrame):
            df = pd.DataFrame(df)

        # Engineer time-based cyclical features using the 24-hour cycle.
        # 'Time' is elapsed seconds since the first transaction in the dataset.
        seconds_in_day = 24 * 60 * 60
        if "Time" in df.columns:
            t = df["Time"].astype(float)
            day_fraction = (t % seconds_in_day) / seconds_in_day
            df["time_sin"] = np.sin(2 * np.pi * day_fraction)
            df["time_cos"] = np.cos(2 * np.pi * day_fraction)
            df["time_log1p"] = np.log1p(t)

        if "Amount" in df.columns:
            amt = df["Amount"].astype(float)
            df["amount_log1p"] = np.log1p(np.maximum(amt, 0.0))
            df["amount_is_high"] = (amt > amt.quantile(0.95)).astype(int)
            df["amount_is_very_high"] = (amt > amt.quantile(0.99)).astype(int)

        # A few simple interaction-style features.
        if "Time" in df.columns and "Amount" in df.columns:
            df["amount_per_second"] = df["Amount"].astype(float) / (df["Time"].astype(float) + 1.0)

        return df


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    expected = {"Time", "Amount", "Class"}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f"Dataset missing columns: {sorted(missing)}")
    return df


def make_splits(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, pd.Series]:
    X = df.drop(columns=["Class"])
    y = df["Class"].astype(int)

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def build_models() -> Dict[str, Any]:
    # Logistic Regression baseline with class-weighting.
    lr = ImbPipeline(
        steps=[
            ("feat", FraudFeatureEngineer()),
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    solver="lbfgs",
                    n_jobs=None,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    # SMOTE + Logistic Regression for explicit imbalance handling.
    smote_lr = ImbPipeline(
        steps=[
            ("feat", FraudFeatureEngineer()),
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    solver="lbfgs",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    # Tree-based models do not need scaling; use class weights.
    rf = ImbPipeline(
        steps=[
            ("feat", FraudFeatureEngineer()),
            ("impute", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    brf = ImbPipeline(
        steps=[
            ("feat", FraudFeatureEngineer()),
            ("impute", SimpleImputer(strategy="median")),
            (
                "model",
                BalancedRandomForestClassifier(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=2,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    return {
        "logreg_balanced": lr,
        "smote_logreg": smote_lr,
        "random_forest_balanced": rf,
        "balanced_random_forest": brf,
    }


def threshold_from_validation(y_true: np.ndarray, y_prob: np.ndarray, fn_cost: float = 10.0, fp_cost: float = 1.0) -> Tuple[float, Dict[str, float]]:
    """Choose threshold minimizing a simple cost-sensitive objective."""
    thresholds = np.unique(np.concatenate(([0.0], y_prob, [1.0])))
    best = {"threshold": 0.5, "cost": float("inf"), "precision": 0.0, "recall": 0.0, "f1": 0.0}
    for thr in thresholds:
        pred = (y_prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        cost = fn_cost * fn + fp_cost * fp
        if cost < best["cost"]:
            best = {
                "threshold": float(thr),
                "cost": float(cost),
                "precision": float(precision_score(y_true, pred, zero_division=0)),
                "recall": float(recall_score(y_true, pred, zero_division=0)),
                "f1": float(f1_score(y_true, pred, zero_division=0)),
            }
    return best["threshold"], best


def evaluate(y_true: np.ndarray, y_prob: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "threshold": float(threshold),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
    }


def cross_validate_models(models: Dict[str, Any], X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    scoring = {
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision",
        "f1": "f1",
        "recall": "recall",
        "precision": "precision",
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for name, model in models.items():
        scores = cross_validate(model, X, y, scoring=scoring, cv=cv, n_jobs=-1, return_train_score=False)
        rows.append(
            {
                "model": name,
                "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
                "pr_auc_mean": float(np.mean(scores["test_pr_auc"])),
                "f1_mean": float(np.mean(scores["test_f1"])),
                "recall_mean": float(np.mean(scores["test_recall"])),
                "precision_mean": float(np.mean(scores["test_precision"])),
            }
        )
    return pd.DataFrame(rows).sort_values(["pr_auc_mean", "roc_auc_mean"], ascending=False).reset_index(drop=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, default="creditcard.csv")
    parser.add_argument("--outdir", type=str, default="artifacts")
    parser.add_argument("--fn-cost", type=float, default=10.0)
    parser.add_argument("--fp-cost", type=float, default=1.0)
    args = parser.parse_args()

    data_path = Path(args.data)
    outdir = Path(args.outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    df = load_data(data_path)

    X_train, X_val, X_test, y_train, y_val, y_test = make_splits(df)
    models = build_models()

    # Cross-validate on the training split to compare candidates.
    cv_results = cross_validate_models(models, X_train, y_train)
    cv_results.to_csv(outdir / "cv_results.csv", index=False)

    best_name = cv_results.iloc[0]["model"]
    best_model = models[best_name]

    # Fit on train split only, tune threshold on validation split.
    best_model.fit(X_train, y_train)
    val_prob = best_model.predict_proba(X_val)[:, 1]
    tuned_threshold, tune_info = threshold_from_validation(
        y_val.to_numpy(), val_prob, fn_cost=args.fn_cost, fp_cost=args.fp_cost
    )

    test_prob = best_model.predict_proba(X_test)[:, 1]
    test_metrics = evaluate(y_test.to_numpy(), test_prob, tuned_threshold)

    # Also provide standard threshold=0.5 metrics for comparison.
    standard_metrics = evaluate(y_test.to_numpy(), test_prob, 0.5)

    artifact = {
        "best_model_name": best_name,
        "threshold": tuned_threshold,
        "tuning": tune_info,
        "test_metrics_tuned": test_metrics,
        "test_metrics_0p5": standard_metrics,
        "dataset_shape": list(df.shape),
        "class_counts": df["Class"].value_counts().to_dict(),
        "feature_columns": [c for c in df.columns if c != "Class"],
    }

    # Save model and metrics.
    joblib.dump(best_model, outdir / "fraud_detector.joblib")
    (outdir / "metrics.json").write_text(json.dumps(artifact, indent=2))
    cv_results.to_json(outdir / "cv_results.json", orient="records", indent=2)

    print("\n=== Cross-validation results ===")
    print(cv_results.to_string(index=False))
    print("\n=== Validation threshold tuning ===")
    print(json.dumps(tune_info, indent=2))
    print("\n=== Test metrics (tuned threshold) ===")
    print(json.dumps({k: v for k, v in test_metrics.items() if k != "classification_report"}, indent=2))
    print("\n=== Test classification report ===")
    print(test_metrics["classification_report"])

    print(f"\nSaved model to: {outdir / 'fraud_detector.joblib'}")
    print(f"Saved metrics to: {outdir / 'metrics.json'}")


if __name__ == "__main__":
    main()
